# Simulación del estómago — Fase 1: modelo químico (sin flujo)

**Objetivo de la fase.** Modelar la química del contenido gástrico como un sistema *perfectamente mezclado* (modelo 0D: sin espacio, solo tiempo). Las piezas son:

1. La secreción de ácido y su neutralización por el tampón de la comida.
2. La activación del pepsinógeno a pepsina.
3. La digestión de la proteína: enzimática (pepsina) y ácida directa.

**Qué hace el cuaderno.**
- Escribe explícitamente las **leyes y ecuaciones gobernantes**.
- Implementa tres integradores propios: Euler, Heun (RK2) y RK4. Se programan a mano para poder medir su error y su orden.
- **Verifica** cada proceso aislado (tests T1–T4) contra su **solución analítica exacta**, con estudio de convergencia.
- Simula el **modelo acoplado** y hace seguimiento del error sin solución exacta. Para ello usa leyes de conservación, autoconvergencia, una solución de referencia y un análisis de estabilidad (rigidez).

**Cómo usarlo en Colab.** `Entorno de ejecución → Ejecutar todo`. No necesita instalar nada: usa NumPy, SciPy, Matplotlib y pandas, que ya vienen en Colab. Tarda menos de un minuto.

> ⚠️ **Sobre los parámetros.** Los valores numéricos son **órdenes de magnitud ilustrativos**, elegidos para que la dinámica sea fisiológicamente plausible: el pH sube tras la comida y luego cae, y la pepsina se activa a pH bajo. **No están ajustados a datos experimentales.** La calibración contra literatura es la fase de *validación*, que viene después. Aquí solo hacemos *verificación*: comprobamos que resolvemos bien las ecuaciones que hemos escrito.

## 1. Leyes y ecuaciones gobernantes

### 1.1 Hipótesis del modelo
| # | Hipótesis | Consecuencia | Se relaja en… |
|---|---|---|---|
| H1 | Mezcla perfecta (0D) | Las concentraciones solo dependen de $t$ | Fase 3 (transporte) |
| H2 | Volumen constante $V$ | Sin vaciado ni dilución por secreción | Fase de vaciado gástrico |
| H3 | Ley de acción de masas para todas las reacciones elementales | Velocidades proporcionales al producto de concentraciones | — |
| H4 | Tampón de la comida modelado como reacción **irreversible** $\mathrm{H^+ + B \to HB}$ | Neutralización simple; no hay equilibrio con $pK_a$ | Mejora futura: tampón en equilibrio |
| H5 | Secreción ácida constante $r_s$ | Sin regulación (gastrina, retroalimentación por pH) | Mejora futura |
| H6 | Actividad de la pepsina modulada por una campana gaussiana en pH | Óptimo cerca de pH 2; casi inactiva por encima de 5 | — |
| H7 | La proteína es un único pseudocomponente $S$, que pasa a péptidos $P$ | No se distinguen enlaces ni tamaños | Fase de partículas |

### 1.2 Especies (variables de estado)
| Símbolo | Significado | Unidades |
|---|---|---|
| $H$ | Concentración de protones $\mathrm{H^+}$ | mM (= mol/m³) |
| $B$ | Capacidad tampón libre de la comida | mM |
| $G$ | Pepsinógeno (zimógeno inactivo) | mM |
| $E$ | Pepsina activa | mM |
| $S$ | Proteína sin digerir | g/L |
| $P$ | Péptidos (proteína digerida) | g/L |

El tiempo $t$ se mide en minutos. El pH se obtiene de $H$ (en mM) mediante la definición:

$$\mathrm{pH} = -\log_{10}\!\left(\frac{H}{1000\ \mathrm{mM/M}}\right) = 3 - \log_{10} H .$$

### 1.3 Velocidades de reacción (ley de acción de masas + cinética enzimática)

**(R1) Neutralización por el tampón**, de segundo orden:
$$ r_b = k_b\, H\, B $$

**(R2) Activación del pepsinógeno**. Es catalizada por el ácido y además **autocatalítica**: la pepsina activa a más pepsinógeno.
$$ r_a = \left(k_{a1}\, H + k_{a2}\, H\, E\right) G $$

**(R3) Proteólisis enzimática**, según Michaelis–Menten con un factor de actividad dependiente del pH:
$$ r_e = k_{cat}\, E\, \phi(\mathrm{pH})\, \frac{S}{K_M + S}, \qquad \phi(\mathrm{pH}) = \exp\!\left[-\left(\frac{\mathrm{pH}-\mathrm{pH}_{opt}}{\sigma}\right)^2\right] $$

**(R4) Hidrólisis ácida directa**, sin enzima. Es mucho más lenta que la enzimática:
$$ r_h = k_H\, H\, S $$

### 1.4 Balances de materia: el sistema de EDOs
Para cada especie se cumple que la variación en el tiempo es igual a lo que se produce menos lo que se consume:

$$
\boxed{
\begin{aligned}
\frac{dH}{dt} &= r_s - r_b \\
\frac{dB}{dt} &= -\,r_b \\
\frac{dG}{dt} &= -\,r_a \\
\frac{dE}{dt} &= +\,r_a \\
\frac{dS}{dt} &= -\,r_e - r_h \\
\frac{dP}{dt} &= +\,r_e + r_h
\end{aligned}}
$$

### 1.5 Leyes de conservación (invariantes exactos del sistema)
Al sumar ecuaciones se cancelan los términos de reacción. Quedan tres cantidades que la solución exacta conserva:

$$
\begin{aligned}
\text{(C1) enzima total:}\quad & G(t) + E(t) = G_0 + E_0 \\
\text{(C2) proteína total:}\quad & S(t) + P(t) = S_0 + P_0 \\
\text{(C3) balance de protones:}\quad & H(t) + \big(B_0 - B(t)\big) - r_s\, t = H_0
\end{aligned}
$$

Son **lineales** en las variables de estado. Esto tiene una consecuencia numérica importante, que comprobaremos en la sección 5.2: los métodos Runge–Kutta conservan *exactamente* cualquier invariante lineal, salvo error de redondeo. Por tanto, para RK estos invariantes **no miden la precisión**. Su papel es **detectar errores de programación**, como un signo cambiado o un término olvidado en las ecuaciones.

## 2. Configuración, parámetros e integradores

In [ ]:
import math
from dataclasses import dataclass, fields

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.special import lambertw

# --- Estilo gráfico: marcas finas, rejilla discreta, colores fijos por método ---
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
})
COLOR = {                      # el color sigue a la entidad, siempre el mismo
    "Euler":      "#2a78d6",   # azul
    "Heun (RK2)": "#eb6834",   # naranja
    "RK4":        "#1baf7a",   # aguamarina
    "exacta":     "#3b3b3b",   # gris oscuro: solución exacta / referencia
}
MARCA = {"Euler": "o", "Heun (RK2)": "s", "RK4": "^"}   # codificación redundante (no solo color)
ORDEN_TEORICO = {"Euler": 1, "Heun (RK2)": 2, "RK4": 4}

pd.set_option("display.float_format", lambda x: f"{x:.3e}")
VERIFICACION = []   # registro de todos los tests: se resume al final

In [ ]:
@dataclass
class Parametros:
    # --- Ácido y tampón ---
    r_s:   float = 0.35     # mM/min        secreción de H+ (~21 mM/h en el contenido)
    k_b:   float = 1.0      # 1/(mM·min)    neutralización H+ + B -> HB
    # --- Activación de pepsina ---
    k_a1:  float = 1.7e-3   # 1/(mM·min)    activación catalizada por ácido
    k_a2:  float = 0.83     # 1/(mM²·min)   activación autocatalítica (por pepsina)
    # --- Proteólisis ---
    k_cat: float = 15.0     # (g/L)/(mM·min) velocidad catalítica por mM de pepsina
    K_M:   float = 5.0      # g/L           constante de Michaelis
    pH_opt: float = 2.0     # —             pH óptimo de la pepsina
    sigma: float = 1.5      # —             anchura de la campana de actividad
    k_H:   float = 1e-5     # 1/(mM·min)    hidrólisis ácida directa
    # --- Condiciones iniciales (justo después de la comida) ---
    H0: float = 0.01        # mM   (pH 5)
    B0: float = 45.0        # mM   capacidad tampón de la comida
    G0: float = 0.02        # mM   pepsinógeno (~20 µM)
    E0: float = 0.0         # mM
    S0: float = 30.0        # g/L  proteína (~25 g en ~0.8 L)
    P0: float = 0.0         # g/L

    def y0(self):
        return np.array([self.H0, self.B0, self.G0, self.E0, self.S0, self.P0])

UNIDADES = {"r_s": "mM/min", "k_b": "1/(mM·min)", "k_a1": "1/(mM·min)", "k_a2": "1/(mM²·min)",
            "k_cat": "(g/L)/(mM·min)", "K_M": "g/L", "pH_opt": "—", "sigma": "—", "k_H": "1/(mM·min)",
            "H0": "mM", "B0": "mM", "G0": "mM", "E0": "mM", "S0": "g/L", "P0": "g/L"}
NOMBRES = ["H", "B", "G", "E", "S", "P"]
p = Parametros()
pd.DataFrame([{"parámetro": f.name, "valor": getattr(p, f.name), "unidades": UNIDADES[f.name],
               "estado": "ilustrativo"} for f in fields(p)]).style.format({"valor": "{:g}"}).hide(axis="index")

### 2.1 Integradores de paso fijo
Para $\dot y = f(t, y)$ con paso $\Delta t$:

| Método | Fórmula | Orden global teórico $p$ |
|---|---|---|
| Euler explícito | $y_{n+1} = y_n + \Delta t\, f(t_n, y_n)$ | 1 |
| Heun (RK2) | $k_1 = f(t_n,y_n),\ k_2 = f(t_n+\Delta t,\, y_n+\Delta t k_1),\ y_{n+1} = y_n + \tfrac{\Delta t}{2}(k_1+k_2)$ | 2 |
| RK4 clásico | $y_{n+1} = y_n + \tfrac{\Delta t}{6}(k_1+2k_2+2k_3+k_4)$ | 4 |

**Orden de convergencia.** Si el error global se comporta como $e(\Delta t) \approx C\,\Delta t^{\,p}$, el orden **observado** entre dos pasos es

$$ p_{obs} = \frac{\ln\big(e(\Delta t_1)/e(\Delta t_2)\big)}{\ln(\Delta t_1/\Delta t_2)} . $$

Que $p_{obs}$ coincida con $p$ teórico es la prueba estándar de que el código implementa correctamente el método y las ecuaciones.

**Normas de error.** Dada la solución numérica $y_n$ y la exacta $y(t_n)$, usamos el error **relativo** (normalizado por la escala de cada variable) para poder comparar especies con unidades distintas:

$$ e_{\infty} = \max_n \frac{|y_n - y(t_n)|}{\max_n |y(t_n)|}, \qquad e_{2} = \sqrt{\frac{1}{N}\sum_n \left(\frac{y_n - y(t_n)}{\max_n|y(t_n)|}\right)^2 } . $$

In [ ]:
def integrar(f, y0, t_final, dt, metodo):
    # Integra dy/dt = f(t, y) desde 0 hasta t_final con paso fijo dt.
    n = int(round(t_final / dt))
    assert abs(n * dt - t_final) < 1e-9 * t_final, "t_final debe ser múltiplo de dt"
    t = np.linspace(0.0, t_final, n + 1)
    y = np.empty((n + 1, len(y0)))
    y[0] = y0
    for i in range(n):
        ti, yi = t[i], y[i]
        if metodo == "Euler":
            y[i + 1] = yi + dt * f(ti, yi)
        elif metodo == "Heun (RK2)":
            k1 = f(ti, yi)
            k2 = f(ti + dt, yi + dt * k1)
            y[i + 1] = yi + 0.5 * dt * (k1 + k2)
        elif metodo == "RK4":
            k1 = f(ti, yi)
            k2 = f(ti + 0.5 * dt, yi + 0.5 * dt * k1)
            k3 = f(ti + 0.5 * dt, yi + 0.5 * dt * k2)
            k4 = f(ti + dt, yi + dt * k3)
            y[i + 1] = yi + dt / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
        else:
            raise ValueError(metodo)
    return t, y


def error_relativo(y_num, y_ref):
    # Errores relativos e_inf y e_2, normalizados por la escala de cada componente.
    escala = np.maximum(np.max(np.abs(y_ref), axis=0), 1e-300)
    e = (y_num - y_ref) / escala
    return np.max(np.abs(e)), np.sqrt(np.mean(e ** 2))


def ejes_dt(ax, dts):
    # Marcas del eje x en los dt realmente usados (evita etiquetas solapadas en log).
    ax.minorticks_off()
    ax.set_xticks(list(dts))
    ax.set_xticklabels([f"{d:g}" for d in dts])


def estudio_convergencia(nombre, f, y0, t_final, exacta, dts):
    # Ejecuta los 3 métodos con cada dt, mide el error y el orden observado.
    filas = []
    for metodo in ORDEN_TEORICO:
        prev = None
        for dt in dts:
            t, y = integrar(f, y0, t_final, dt, metodo)
            e_inf, e_2 = error_relativo(y, exacta(t))
            p_obs = np.nan if prev is None else math.log(prev[1] / e_inf) / math.log(prev[0] / dt)
            filas.append({"método": metodo, "dt": dt, "e_inf": e_inf, "e_2": e_2, "orden_obs": p_obs})
            prev = (dt, e_inf)
    df = pd.DataFrame(filas)
    # Criterio de verificación: el orden observado en los dos pasos más finos
    # debe estar a ±0.25 del teórico.
    for metodo, p_teo in ORDEN_TEORICO.items():
        p_fin = df[df["método"] == metodo]["orden_obs"].iloc[-2:].mean()
        VERIFICACION.append({"test": nombre, "método": metodo, "orden_teórico": p_teo,
                             "orden_observado": p_fin, "pasa": abs(p_fin - p_teo) < 0.25})
    return df


def grafica_test(titulo, f, y0, t_final, exacta, df, dt_muestra, componente=0, etiqueta_y=""):
    # Izquierda: exacta vs numérica (dt grueso). Derecha: error vs dt (log-log) con pendientes teóricas.
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    tt = np.linspace(0, t_final, 600)
    a1.plot(tt, exacta(tt)[:, componente], color=COLOR["exacta"], lw=2.5, label="exacta")
    for metodo in ORDEN_TEORICO:
        t, y = integrar(f, y0, t_final, dt_muestra, metodo)
        a1.plot(t, y[:, componente], ls="none", marker=MARCA[metodo], ms=5, mfc="none", mew=1.2,
                color=COLOR[metodo], label=f"{metodo}")
    a1.set(xlabel="t [min]", ylabel=etiqueta_y, title=f"Solución (Δt = {dt_muestra:g} min)")
    a1.legend()

    for metodo, p_teo in ORDEN_TEORICO.items():
        d = df[df["método"] == metodo]
        a2.loglog(d["dt"], d["e_inf"], marker=MARCA[metodo], ms=6, color=COLOR[metodo],
                  label=f"{metodo}  (p_obs ≈ {d['orden_obs'].iloc[-2:].mean():.2f})")
        # recta guía con pendiente teórica, anclada al punto más fino
        x = np.array([d["dt"].min(), d["dt"].max()])
        a2.loglog(x, d["e_inf"].iloc[-1] * (x / x[0]) ** p_teo, ls=":", lw=1.2, color=COLOR[metodo])
    ejes_dt(a2, sorted(df["dt"].unique()))
    a2.set(xlabel="Δt [min]", ylabel="error relativo $e_\\infty$",
           title="Convergencia (punteado = pendiente teórica)")
    a2.legend(loc="lower right")
    fig.suptitle(titulo, fontweight="bold")
    fig.tight_layout()
    plt.show()

## 3. Verificación proceso a proceso (soluciones exactas)

Cada test aísla un proceso del modelo completo, fijando el resto de variables, y lo compara con su solución analítica.

### T1 — Hidrólisis ácida a pH constante (primer orden)
Con $H$ constante, la reacción R4 da una cinética de primer orden:
$$ \frac{dS}{dt} = -k\,S,\quad k = k_H H \quad\Longrightarrow\quad S(t) = S_0\, e^{-k t}. $$
Para que el test sea exigente usamos un $k$ efectivo mayor (0.05 min⁻¹) que el fisiológico.

In [ ]:
k1 = 0.05                                   # 1/min  (constante efectiva del test)
f_T1 = lambda t, y: np.array([-k1 * y[0]])
y0_T1 = np.array([p.S0])
exacta_T1 = lambda t: (p.S0 * np.exp(-k1 * np.asarray(t)))[:, None]

dts_T1 = [8, 4, 2, 1, 0.5]
df_T1 = estudio_convergencia("T1 primer orden", f_T1, y0_T1, 120, exacta_T1, dts_T1)
grafica_test("T1 · Hidrólisis ácida de primer orden", f_T1, y0_T1, 120, exacta_T1, df_T1,
             dt_muestra=8, etiqueta_y="S [g/L]")
df_T1

### T2 — Proteólisis de Michaelis–Menten con pepsina y pH constantes
Con $E$ y pH fijos, R3 queda $\dfrac{dS}{dt} = -\dfrac{V_{max}\,S}{K_M+S}$ con $V_{max} = k_{cat} E\,\phi$. Separando variables e integrando:
$$ K_M \ln\frac{S}{S_0} + (S - S_0) = -V_{max}\, t . $$
Esta relación implícita se despeja con la **función W de Lambert** (la inversa de $w\,e^{w}$):
$$ S(t) = K_M\; W\!\left( \frac{S_0}{K_M}\, \exp\!\left(\frac{S_0 - V_{max}\, t}{K_M}\right)\right). $$

In [ ]:
Vmax = p.k_cat * p.G0 * 1.0           # pepsina totalmente activa, pH óptimo (phi = 1) -> 0.3 g/L/min
f_T2 = lambda t, y: np.array([-Vmax * y[0] / (p.K_M + y[0])])
y0_T2 = np.array([p.S0])
def exacta_T2(t):
    arg = (p.S0 / p.K_M) * np.exp((p.S0 - Vmax * np.asarray(t)) / p.K_M)
    return (p.K_M * np.real(lambertw(arg)))[:, None]

dts_T2 = [16, 8, 4, 2, 1]
df_T2 = estudio_convergencia("T2 Michaelis–Menten", f_T2, y0_T2, 240, exacta_T2, dts_T2)
grafica_test("T2 · Proteólisis Michaelis–Menten (solución con W de Lambert)",
             f_T2, y0_T2, 240, exacta_T2, df_T2, dt_muestra=16, etiqueta_y="S [g/L]")
df_T2

### T3 — Neutralización ácido–tampón (segundo orden, sin secreción)
Con $r_s = 0$: $\dot H = \dot B = -k_b H B$. La diferencia $\Delta = H - B = H_0 - B_0$ se conserva. Sustituyendo $B = H - \Delta$ se obtiene una ecuación logística, cuya solución es
$$ H(t) = \frac{\Delta}{1 - \dfrac{B_0}{H_0}\, e^{-k_b \Delta\, t}}, \qquad B(t) = H(t) - \Delta . $$
Usamos un ácido inicial de 40 mM contra 50 mM de tampón: el ácido se agota y queda tampón sobrante. Para ver una dinámica de varios minutos usamos $k_b = 0.01$.

In [ ]:
kb_T3, H0_T3, B0_T3 = 0.01, 40.0, 50.0
D = H0_T3 - B0_T3
f_T3 = lambda t, y: np.array([-kb_T3 * y[0] * y[1], -kb_T3 * y[0] * y[1]])
y0_T3 = np.array([H0_T3, B0_T3])
def exacta_T3(t):
    H = D / (1.0 - (B0_T3 / H0_T3) * np.exp(-kb_T3 * D * np.asarray(t)))
    return np.column_stack([H, H - D])

dts_T3 = [1, 0.5, 0.25, 0.125, 0.0625]
df_T3 = estudio_convergencia("T3 neutralización 2º orden", f_T3, y0_T3, 30, exacta_T3, dts_T3)
grafica_test("T3 · Neutralización H⁺ + B → HB", f_T3, y0_T3, 30, exacta_T3, df_T3,
             dt_muestra=1, componente=0, etiqueta_y="H [mM]")
df_T3

### T4 — Activación autocatalítica del pepsinógeno a pH constante
Con $H$ fijo, R2 da $\dot E = (a + bE)(T - E)$, donde $a = k_{a1}H$, $b = k_{a2}H$ y $T = G_0 + E_0$ es la enzima total (ley C1). Por fracciones parciales:
$$ \frac{a + bE}{T - E} = R(t) := \frac{a + bE_0}{T - E_0}\, e^{(a + bT)\,t} \quad\Longrightarrow\quad E(t) = \frac{R(t)\,T - a}{b + R(t)}. $$
La curva tiene forma sigmoidea: arranca lenta (solo actúa el ácido) y se acelera cuando hay pepsina suficiente para autocatalizar. Usamos pH 1.5.

In [ ]:
H_T4 = 10 ** (3 - 1.5)                 # mM a pH 1.5
a_, b_, T_ = p.k_a1 * H_T4, p.k_a2 * H_T4, p.G0 + p.E0
f_T4 = lambda t, y: np.array([(a_ + b_ * y[0]) * (T_ - y[0])])
y0_T4 = np.array([p.E0])
def exacta_T4(t):
    R = (a_ + b_ * p.E0) / (T_ - p.E0) * np.exp((a_ + b_ * T_) * np.asarray(t))
    return ((R * T_ - a_) / (b_ + R))[:, None]

dts_T4 = [2, 1, 0.5, 0.25, 0.125]
df_T4 = estudio_convergencia("T4 activación autocatalítica", f_T4, y0_T4, 30, exacta_T4, dts_T4)
grafica_test("T4 · Activación autocatalítica del pepsinógeno (pH 1.5)", f_T4, y0_T4, 30, exacta_T4, df_T4,
             dt_muestra=2, etiqueta_y="E [mM]")
df_T4

## 4. Modelo químico completo (acoplado)

Juntamos las cuatro reacciones con secreción continua de ácido. En el modelo completo no hay solución exacta; la referencia es un integrador implícito de alta precisión (`Radau`, de orden 5, adecuado para problemas rígidos). Además, la contrastamos con un segundo integrador independiente (`DOP853`, de orden 8), para estimar cuánto podemos fiarnos de la propia referencia.

In [ ]:
def make_rhs(p, bug=None):
    # Lado derecho del sistema de EDOs (sección 1.4).
    # `bug` permite inyectar un error a propósito (sección 5.2).
    def f(t, y):
        H, B, G, E, S, P = y
        pH = 3.0 - math.log10(max(H, 1e-12))
        phi = math.exp(-((pH - p.pH_opt) / p.sigma) ** 2)
        r_b = p.k_b * H * B                                   # R1
        r_a = (p.k_a1 * H + p.k_a2 * H * E) * G               # R2
        r_e = p.k_cat * E * phi * S / (p.K_M + S)             # R3
        r_h = p.k_H * H * S                                   # R4
        dP = r_e + r_h if bug != "olvido_rh_en_P" else r_e    # bug: término olvidado
        return np.array([p.r_s - r_b, -r_b, -r_a, r_a, -r_e - r_h, dP])
    return f

def pH_de(H):
    return 3.0 - np.log10(np.maximum(H, 1e-12))

T_FINAL = 240.0                                 # min (4 h de digestión)
f_full = make_rhs(p)
ref  = solve_ivp(f_full, (0, T_FINAL), p.y0(), method="Radau",  rtol=1e-12, atol=1e-14, dense_output=True)
ref2 = solve_ivp(f_full, (0, T_FINAL), p.y0(), method="DOP853", rtol=1e-13, atol=1e-15, dense_output=True)
assert ref.success and ref2.success
t_plot = np.linspace(0, T_FINAL, 2401)
Y = ref.sol(t_plot).T
inc_ref = error_relativo(ref2.sol(t_plot).T, Y)[0]
print(f"Pasos Radau: {ref.t.size}   Pasos DOP853: {ref2.t.size}")
print(f"Incertidumbre de la referencia (Radau vs DOP853, e_inf relativo): {inc_ref:.1e}")

In [ ]:
H, B, G, E, S, P = Y.T
pH = pH_de(H)
phi = np.exp(-((pH - p.pH_opt) / p.sigma) ** 2)

# Instantes característicos
t_tampon = t_plot[np.argmax(B < 0.01 * p.B0)]           # tampón agotado al 99 %
t_E50    = t_plot[np.argmax(E > 0.5 * (p.G0 + p.E0))]   # mitad de la pepsina activa
t_S50    = t_plot[np.argmax(S < 0.5 * p.S0)]            # mitad de la proteína digerida

fig, ax = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
a = ax[0, 0]
a.plot(t_plot, pH, color=COLOR["exacta"]); a.set(ylabel="pH", title="Acidez del contenido")
a.axhspan(p.pH_opt - 0.5, p.pH_opt + 0.5, color="#1baf7a", alpha=0.12, lw=0)
a.text(4, p.pH_opt, "zona óptima pepsina", va="center", fontsize=9, color="#52514e")
a.invert_yaxis()

a = ax[0, 1]
a.plot(t_plot, B, color="#2a78d6", label="tampón libre B")
a.plot(t_plot, H, color="#eb6834", label="H⁺ libre")
a.set(ylabel="mM", title="Ácido frente a tampón"); a.legend()

a = ax[1, 0]
a.plot(t_plot, 1e3 * G, color="#2a78d6", label="pepsinógeno G")
a.plot(t_plot, 1e3 * E, color="#eb6834", label="pepsina activa E")
a2 = a.twinx(); a2.plot(t_plot, phi, color="#6250d6", lw=1.2, ls="--"); a2.set_ylim(0, 1.05)
a2.set_ylabel("actividad φ(pH)  (línea discontinua)", color="#52514e"); a2.grid(False)
a.set(xlabel="t [min]", ylabel="µM", title="Activación de la enzima"); a.legend(loc="center left")

a = ax[1, 1]
a.plot(t_plot, S, color="#2a78d6", label="proteína S")
a.plot(t_plot, P, color="#eb6834", label="péptidos P")
a.set(xlabel="t [min]", ylabel="g/L", title="Digestión de proteína"); a.legend()

for a in ax.flat:
    for tc in (t_tampon, t_E50, t_S50):
        a.axvline(tc, color="#9a9a96", lw=0.8, ls=":")
fig.suptitle("Modelo químico completo (referencia Radau)", fontweight="bold")
fig.tight_layout(); plt.show()

print(f"Tampón agotado (99 %):      t ≈ {t_tampon:6.1f} min")
print(f"50 % de la pepsina activa:  t ≈ {t_E50:6.1f} min")
print(f"50 % de la proteína digerida: t ≈ {t_S50:6.1f} min")

**Lectura de la dinámica.** La comida tampona el ácido secretado y el pH se mantiene alto. Mientras dura esa fase, la pepsina ni se activa ni trabaja. Cuando el tampón se agota, el pH cae en pocos minutos. Entonces se dispara la activación autocatalítica y la proteólisis arranca. Este comportamiento de "retardo y luego arranque" **emerge** del acoplamiento de las ecuaciones; no se ha impuesto a mano. Las líneas verticales punteadas marcan los tres instantes característicos.

## 5. Seguimiento del error en el modelo acoplado

Sin solución exacta, el error se vigila con cuatro herramientas complementarias:

| Herramienta | Qué detecta |
|---|---|
| 5.1 Error frente a referencia de alta precisión | Magnitud del error y orden de convergencia |
| 5.2 Leyes de conservación C1–C3 | Errores de programación en las ecuaciones |
| 5.3 Autoconvergencia (Richardson) | Orden observado **sin usar ninguna referencia** |
| 5.4 Positividad y estabilidad (rigidez) | Pasos de tiempo que hacen explotar la solución |

### 5.1 Error frente a la referencia

In [ ]:
dts_full = [0.02, 0.01, 0.005, 0.0025]
res_full = {}                                   # guardamos las soluciones para 5.2 y 5.3
filas = []
for metodo in ORDEN_TEORICO:
    prev = None
    for dt in dts_full:
        t, y = integrar(f_full, p.y0(), T_FINAL, dt, metodo)
        res_full[(metodo, dt)] = (t, y)
        e_inf, e_2 = error_relativo(y, ref.sol(t).T)
        p_obs = np.nan if prev is None else math.log(prev[1] / e_inf) / math.log(prev[0] / dt)
        filas.append({"método": metodo, "dt": dt, "e_inf": e_inf, "e_2": e_2, "orden_obs": p_obs})
        prev = (dt, e_inf)
df_full = pd.DataFrame(filas)
for metodo, p_teo in ORDEN_TEORICO.items():
    p_fin = df_full[df_full["método"] == metodo]["orden_obs"].iloc[-2:].mean()
    VERIFICACION.append({"test": "Modelo completo vs Radau", "método": metodo, "orden_teórico": p_teo,
                         "orden_observado": p_fin, "pasa": abs(p_fin - p_teo) < 0.25})

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
for metodo, p_teo in ORDEN_TEORICO.items():
    d = df_full[df_full["método"] == metodo]
    a1.loglog(d["dt"], d["e_inf"], marker=MARCA[metodo], color=COLOR[metodo],
              label=f"{metodo}  (p_obs ≈ {d['orden_obs'].iloc[-2:].mean():.2f})")
    x = np.array([d["dt"].min(), d["dt"].max()])
    a1.loglog(x, d["e_inf"].iloc[-1] * (x / x[0]) ** p_teo, ls=":", lw=1.2, color=COLOR[metodo])
a1.axhline(inc_ref, color=COLOR["exacta"], lw=1, ls="--")
a1.text(min(dts_full), inc_ref * 1.8, "incertidumbre de la referencia", ha="left", fontsize=9, color="#52514e")
ejes_dt(a1, sorted(dts_full))
a1.set(xlabel="Δt [min]", ylabel="error relativo $e_\\infty$", title="Convergencia del modelo completo")
a1.legend(loc="lower right", bbox_to_anchor=(1.0, 0.07))

# Evolución temporal del error (dt = 0.02): ¿dónde se acumula?
for metodo in ORDEN_TEORICO:
    t, y = res_full[(metodo, 0.02)]
    yr = ref.sol(t).T
    esc = np.max(np.abs(yr), axis=0)
    a2.semilogy(t, np.max(np.abs(y - yr) / esc, axis=1) + 1e-18, color=COLOR[metodo], lw=1.4, label=metodo)
a2.axvline(t_tampon, color="#9a9a96", lw=0.8, ls=":")
a2.text(t_tampon + 3, 1e-14, "tampón\nagotado", fontsize=9, color="#52514e")
a2.set(xlabel="t [min]", ylabel="error relativo (máx. sobre especies)", title="Error a lo largo del tiempo (Δt = 0.02)")
a2.legend()
fig.tight_layout(); plt.show()
df_full

Dos observaciones:
- **Régimen pre-asintótico.** Los órdenes observados de Heun y RK4 arrancan *por encima* del teórico y bajan hacia él al refinar. La razón es que con estos pasos todavía no se cumple $|\lambda|\,\Delta t \ll 1$ para la reacción más rápida (ver 5.4), así que los términos de orden superior del error aún pesan. Por eso este estudio usa pasos más finos que los tests aislados. Además, no se baja de $\Delta t = 0.0025$: el error de RK4 ya está a menos de un orden de magnitud de la incertidumbre de la propia referencia (línea discontinua), y por debajo el "error" mediría la referencia, no a RK4.
- El error **crece bruscamente cuando se agota el tampón**: es la fase de dinámica más rápida del sistema. Es el comportamiento típico de un problema con escalas de tiempo muy distintas, y anticipa la sección 5.4.

### 5.2 Leyes de conservación como detector de errores de programación
Calculamos el residuo de C1, C2 y C3 a lo largo de la simulación:
- Con las ecuaciones correctas, el residuo debe quedarse en el nivel del **redondeo** (~1e-13), para cualquier método RK.
- Para demostrar su utilidad, inyectamos un error típico de programación: olvidar el término $r_h$ en $dP/dt$.

In [ ]:
def residuos_conservacion(t, y, p):
    H, B, G, E, S, P = y.T
    C1 = (G + E) - (p.G0 + p.E0)
    C2 = (S + P) - (p.S0 + p.P0)
    C3 = (H + (p.B0 - B) - p.r_s * t) - p.H0
    return {"C1 enzima": np.abs(C1) / (p.G0 + p.E0),
            "C2 proteína": np.abs(C2) / (p.S0 + p.P0),
            "C3 protones": np.abs(C3) / (p.B0 + p.r_s * T_FINAL)}

COLOR_C = {"C1 enzima": "#2a78d6", "C2 proteína": "#eb6834", "C3 protones": "#1baf7a"}
t_ok, y_ok = res_full[("RK4", 0.02)]
t_bug, y_bug = integrar(make_rhs(p, bug="olvido_rh_en_P"), p.y0(), T_FINAL, 0.02, "RK4")

fig, axs = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
for a, (t_, y_, tit) in zip(axs, [(t_ok, y_ok, "Código correcto (RK4, Δt = 0.02)"),
                                  (t_bug, y_bug, "Con error inyectado: falta r_h en dP/dt")]):
    for nombre, r in residuos_conservacion(t_, y_, p).items():
        a.semilogy(t_, r + 1e-18, color=COLOR_C[nombre], lw=1.4, label=nombre)
    a.axhline(1e-12, color="#9a9a96", lw=0.8, ls="--")
    a.set(xlabel="t [min]", title=tit)
axs[0].set_ylabel("residuo relativo |C(t) − C(0)|")
axs[0].legend(); axs[0].set_ylim(1e-18, 1)
fig.tight_layout(); plt.show()

tabla_C = []
for metodo in ORDEN_TEORICO:
    t_, y_ = res_full[(metodo, 0.02)]
    tabla_C.append({"método": metodo, **{k: v.max() for k, v in residuos_conservacion(t_, y_, p).items()}})
max_res = max(max(v for k, v in fila.items() if k != "método") for fila in tabla_C)
VERIFICACION.append({"test": "Conservación C1–C3 (todas, Δt = 0.02)", "método": "todos",
                     "orden_teórico": np.nan, "orden_observado": np.nan, "pasa": max_res < 1e-10})
bug_detectado = residuos_conservacion(t_bug, y_bug, p)["C2 proteína"].max() > 1e-6
VERIFICACION.append({"test": "Detección del error inyectado", "método": "RK4",
                     "orden_teórico": np.nan, "orden_observado": np.nan, "pasa": bug_detectado})
pd.DataFrame(tabla_C)

Con el código correcto, los tres residuos se quedan en ~1e-15 para los tres métodos. Esto confirma lo anunciado en 1.5: **los invariantes lineales no miden la precisión de un RK**; un Euler con error del 1 % los conserva igual de bien. Con el error inyectado, en cambio, C2 crece sin parar. Por eso conviene que este test se ejecute **siempre** que se toquen las ecuaciones. Cuando el modelo incluya flujo y vaciado, estos balances se convertirán en balances con entradas y salidas, y seguirán siendo la primera alarma.

### 5.3 Autoconvergencia (sin referencia)
Cuando no existe ni solución exacta ni una referencia fiable, que es lo que pasará con Navier–Stokes, se comparan tres resoluciones entre sí: $\Delta t$, $\Delta t/2$ y $\Delta t/4$. El orden observado es

$$ p_{obs} = \log_2 \frac{\| y_{\Delta t} - y_{\Delta t/2} \|}{\| y_{\Delta t/2} - y_{\Delta t/4}\|}, $$

y la **extrapolación de Richardson** da una estimación del error de la solución más fina:

$$ e_{\Delta t/4} \approx \frac{\| y_{\Delta t/2} - y_{\Delta t/4}\|}{2^{p} - 1}. $$

Las soluciones se comparan en los instantes comunes a las tres mallas. Usamos el trío $\Delta t$ = 0.005 / 0.0025 / 0.00125 para estar en régimen asintótico (con pasos más gruesos, el orden observado sale inflado, como se vio en 5.1).

In [ ]:
def en_malla_gruesa(y_fino, factor):
    return y_fino[::factor]

filas = []
for metodo, p_teo in ORDEN_TEORICO.items():
    _, y1 = res_full[(metodo, 0.005)]
    _, y2 = res_full[(metodo, 0.0025)]
    t4, y4 = integrar(f_full, p.y0(), T_FINAL, 0.00125, metodo)   # malla extra, solo para este estudio
    y2c, y4c = en_malla_gruesa(y2, 2), en_malla_gruesa(y4, 4)
    esc = np.max(np.abs(y4c), axis=0)
    d12 = np.max(np.abs(y1 - y2c) / esc)
    d24 = np.max(np.abs(y2c - y4c) / esc)
    p_obs = math.log2(d12 / d24)
    e_rich = d24 / (2 ** p_teo - 1)
    e_real = error_relativo(y4, ref.sol(t4).T)[0]
    filas.append({"método": metodo, "orden_teórico": p_teo, "orden_obs (sin referencia)": p_obs,
                  "error estimado Richardson (Δt=0.00125)": e_rich, "error real vs Radau": e_real,
                  "estimado / real": e_rich / e_real})
    VERIFICACION.append({"test": "Autoconvergencia (Richardson)", "método": metodo, "orden_teórico": p_teo,
                         "orden_observado": p_obs, "pasa": abs(p_obs - p_teo) < 0.25})
pd.DataFrame(filas)

La columna *estimado / real* cercana a 1 muestra que Richardson **estima bien el error sin conocer la solución**. Es la herramienta que usaremos en las fases con fluido. Una advertencia sobre RK4: su error "real" (~2e-12) ya está por debajo de la incertidumbre de la referencia (~5e-12), así que esa celda mide sobre todo la referencia. En ese caso, la estimación de Richardson es probablemente más fiable que la comparación.

### 5.4 Rigidez y estabilidad
La reacción R1 es mucho más rápida que el resto. Su escala de tiempo, linealizando, la da el autovalor

$$ \lambda_b \approx -k_b\,(B + H), \qquad \tau_b = \frac{1}{|\lambda_b|} \sim \frac{1}{k_b B_0} \approx 0.02\ \text{min}. $$

En cambio, la digestión ocurre en horas. Esta separación de escalas define un problema **rígido**. Los métodos explícitos solo son estables si $|\lambda|\,\Delta t$ cae dentro de su región de estabilidad: sobre el eje real, $|\lambda|\Delta t < 2$ para Euler, $< 2$ para Heun y $< 2.785$ para RK4. Por encima de ese límite, la solución **explota**, por pequeño que sea el error de truncamiento.

In [ ]:
lam = p.k_b * (Y[:, 1] + Y[:, 0])               # |lambda_b(t)| sobre la referencia
lim = {"Euler": 2.0, "Heun (RK2)": 2.0, "RK4": 2.785}
dt_max = {m: (lim[m] / lam).min() for m in lim}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
for m in lim:
    a1.semilogy(t_plot, lim[m] / lam, color=COLOR[m], lw=1.6, ls="--" if m == "Euler" else "-",
                label=f"{m}: {dt_max[m]:.3f} min")
a1.set(xlabel="t [min]", ylabel="Δt máximo estable [min]  (leyenda: mínimo)",
       title="Límite de estabilidad (Euler y Heun coinciden)")
a1.legend(loc="upper left")

# Experimento: RK4 por debajo y por encima del límite, primeros 12 minutos
t_ref12 = t_plot[t_plot <= 12]
a2.plot(t_ref12, pH_de(Y[t_plot <= 12, 0]), color=COLOR["exacta"], lw=3, alpha=0.35, label="referencia")
experimentos = [(0.02, "#1baf7a", "-",  "estable"),
                (0.08, "#6250d6", "--", "inestable: solución espuria"),
                (0.10, "#e34948", ":",  "inestable: diverge")]
err_exp = {}
for dt, col, ls, txt in experimentos:
    with np.errstate(all="ignore"):
        t, y = integrar(f_full, p.y0(), 12.0, dt, "RK4")
        err_exp[dt] = np.nan_to_num(error_relativo(y, ref.sol(t).T)[0], nan=np.inf)
        H_ = np.where(np.isfinite(y[:, 0]) & (y[:, 0] > 0), y[:, 0], np.nan)
    a2.plot(t, pH_de(H_), color=col, ls=ls, lw=1.8, label=f"RK4, Δt = {dt:g}: {txt}")
a2.set_ylim(1, 6.5)
a2.set(xlabel="t [min]", ylabel="pH", title="Primeros 12 min: el paso de tiempo decide")
a2.legend(loc="center right", fontsize=9)
fig.tight_layout(); plt.show()

print("Error relativo e_inf en los primeros 12 min:")
for dt in err_exp:
    print(f"  Δt = {dt:<5g} (|λ|Δt ≈ {lam[0]*dt:.2f})  ->  e_inf = {err_exp[dt]:.2e}")
VERIFICACION.append({"test": "Estabilidad: Δt < límite estable y Δt > límite falla", "método": "RK4",
                     "orden_teórico": np.nan, "orden_observado": np.nan,
                     "pasa": err_exp[0.02] < 1e-2 and err_exp[0.08] > 0.1 and err_exp[0.10] > 0.1})
for m in lim:
    print(f"{m:12s}  Δt máximo estable ≈ {dt_max[m]:.4f} min")

Fíjate en $\Delta t = 0.08$: la solución **no explota**. El método converge a un estado *espurio* con pH ≈ 2.7 en lugar de ≈ 5, un valor que parece fisiológicamente razonable. Es el error más peligroso: un resultado plausible y falso. Solo el análisis de estabilidad o un estudio de convergencia lo delatan.

**Consecuencia práctica.** Con un método explícito, el paso de tiempo lo dicta la reacción más rápida (R1), no la precisión que necesitamos para la digestión. Hay dos salidas, y ambas reaparecerán al acoplar con el fluido:
- **Métodos implícitos** (como Radau o BDF), estables con pasos grandes.
- **Separación de operadores** (*operator splitting*): la química rígida se resuelve con un método implícito y el transporte con uno explícito.

### 5.5 Positividad
Las concentraciones no pueden ser negativas. Comprobamos que ninguna solución estable baja de cero, admitiendo una tolerancia del orden del error numérico.

In [ ]:
filas = []
for (metodo, dt), (t, y) in res_full.items():
    minimo = y.min(axis=0)
    filas.append({"método": metodo, "dt": dt, **{f"min {n}": v for n, v in zip(NOMBRES, minimo)}})
df_pos = pd.DataFrame(filas)
tol = 1e-10
pos_ok = bool((df_pos[[f"min {n}" for n in NOMBRES]] > -tol).all().all())
VERIFICACION.append({"test": "Positividad de concentraciones", "método": "todos",
                     "orden_teórico": np.nan, "orden_observado": np.nan, "pasa": pos_ok})
df_pos

## 6. Resumen de verificación
Todos los tests del cuaderno en una tabla. La última celda **falla con un error** si alguno no pasa. Así, si en el futuro se modifica el modelo y algo se rompe, *Ejecutar todo* lo detecta de inmediato.

In [ ]:
resumen = pd.DataFrame(VERIFICACION)
resumen["resultado"] = np.where(resumen["pasa"], "✅ pasa", "❌ FALLA")
display(resumen.drop(columns="pasa").style.format({"orden_observado": "{:.3f}", "orden_teórico": "{:.0f}"},
                                                  na_rep="—").hide(axis="index"))
n_ok, n = int(resumen["pasa"].sum()), len(resumen)
print(f"\n{n_ok}/{n} tests superados")
assert n_ok == n, "Hay tests de verificación que no pasan: revisar antes de continuar"

## 7. Qué hemos establecido y siguientes pasos

**Establecido en esta fase**
- Las ecuaciones de las cuatro reacciones están implementadas y verificadas una a una contra soluciones exactas. Los tres integradores alcanzan su orden teórico.
- En el modelo acoplado, el error se controla con referencia de alta precisión, autoconvergencia (Richardson), conservación y positividad.
- Hemos identificado la **rigidez** introducida por la neutralización ácido–tampón. Fija el paso máximo en ~0.06 min para RK4 y condicionará el diseño del acoplamiento con el fluido.

**Mejoras del modelo químico, pendientes de validación con literatura**
1. Tampón en **equilibrio** con su $pK_a$, en lugar de reacción irreversible (relaja H4).
2. **Vaciado gástrico** y dilución por la secreción (relaja H2). Los balances C1–C3 pasan a tener términos de salida.
3. **Regulación de la secreción** en función del pH (relaja H5).
4. **Inactivación de la pepsina** a pH neutro.
5. **Calibración**: ajustar los parámetros a curvas publicadas de pH gástrico y de digestión in vitro (protocolos tipo INFOGEST).

**Siguiente fase (2): transporte 1D.** Las mismas especies, pero con advección–difusión–reacción en una dimensión. La química de este cuaderno se reutiliza tal cual como término de reacción.